# 05 · Export Data for the Website

The website draws its charts in the browser, so it needs the **final numbers**, not the full dataset. This notebook calculates the numbers for each section of the story and saves them as small JSON files in `web/public/data/`.

The calculations are the same as in `03_eda`. Run `04_model` first, because the match simulator uses `model_export.json` from it.

In [1]:
import json
import shutil
from pathlib import Path

import pandas as pd

OUT = Path("../web/public/data")
OUT.mkdir(parents=True, exist_ok=True)

all_matches = pd.read_csv("../data/clean/matches.csv", parse_dates=["date"])
df = all_matches[all_matches["crowd_status"] != "unknown"]   # crowd unknown: left out of crowd comparisons

LEAGUES = ["EPL", "La_Liga", "Serie_A", "Bundesliga", "Ligue_1"]

def save(name, data):
    with open(OUT / name, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    print("saved", name)

def by_league(data, fn):
    """Run fn on all leagues together, then on each league."""
    result = {"All": fn(data)}
    for league in LEAGUES:
        result[league] = fn(data[data["league"] == league])
    return result

## Headline numbers (hook section)

In [2]:
summary = {
    "matches": int(len(all_matches)),
    "leagues": int(df["league"].nunique()),
    "seasons": int(df["season"].nunique()),
    "ghost_games": int((df["crowd_status"] == "none").sum()),
    "first_date": str(all_matches["date"].min().date()),
    "last_date": str(all_matches["date"].max().date()),
}
save("summary.json", summary)
summary

saved summary.json


{'matches': 16109,
 'leagues': 5,
 'seasons': 9,
 'ghost_games': 2122,
 'first_date': '2017-08-04',
 'last_date': '2026-05-24'}

## Section 3: attendance over time

Average attendance per match, by month. Ghost games count as 0, so the drop to empty stadiums shows clearly. Months with fewer than 10 matches are left out.

In [3]:
month = df["date"].dt.to_period("M")
monthly = df.groupby([month, "league"])["attendance"].mean().unstack()
counts = df.groupby([month, "league"]).size().unstack()
monthly["All"] = df.groupby(month)["attendance"].mean()
counts["All"] = df.groupby(month).size()

# hide months with fewer than 10 matches (e.g. one leftover match in June), they give misleading averages
monthly = monthly.where(counts >= 10)
monthly = monthly.dropna(how="all")

attendance = []
for month, row in monthly.iterrows():
    item = {"month": str(month)}
    for col in monthly.columns:
        item[col] = None if pd.isna(row[col]) else int(round(row[col]))
    attendance.append(item)

save("attendance.json", attendance)
pd.DataFrame(attendance).tail()

saved attendance.json


,month,Bundesliga,EPL,La_Liga,Ligue_1,Serie_A,All
87,2026-01,40988.0,39960.0,28073.0,26740.0,29200.0,33225
88,2026-02,42346.0,41708.0,30632.0,25241.0,27951.0,33649
89,2026-03,44514.0,41812.0,34096.0,29327.0,28714.0,35471
90,2026-04,42410.0,42419.0,30698.0,27561.0,29681.0,34139
91,2026-05,42557.0,42159.0,30193.0,29261.0,28795.0,34185


## Section 4: results before, during and after ghost games

In [4]:
def results_by_period(data):
    periods = {
        "Before COVID": data[data["date"] < "2020-03-01"],
        "Empty stadiums": data[data["crowd_status"] == "none"],
        "Fans back": data[data["date"] >= "2022-07-01"],
    }
    rows = []
    for name, part in periods.items():
        share = part["result"].value_counts(normalize=True) * 100
        rows.append({
            "period": name,
            "matches": int(len(part)),
            "home_win": round(share.get("H", 0), 1),
            "draw": round(share.get("D", 0), 1),
            "away_win": round(share.get("A", 0), 1),
        })
    return rows

results = by_league(df, results_by_period)
save("results.json", results)
pd.DataFrame(results["All"])

saved results.json


,period,matches,home_win,draw,away_win
0,Before COVID,4904,45.0,25.0,30.0
1,Empty stadiums,2122,39.9,25.3,34.8
2,Fans back,7072,43.8,25.3,31.0


## Section 5: the referee

Average extra fouls, yellow cards and red cards given to the away team (away − home), with fans vs without. Red cards are rare, so they're shown **per 100 matches**.

In [5]:
def referee_stats(data):
    rows = []
    for label, crowd in [("With fans", 1), ("Empty stadiums", 0)]:
        part = data[data["crowd"] == crowd]
        rows.append({
            "group": label,
            "matches": int(len(part)),
            "fouls": round(part["foul_diff"].mean(), 2),
            "yellows": round(part["yellow_diff"].mean(), 2),
            "reds_per_100": round(part["red_diff"].mean() * 100, 1),
        })
    return rows

referee = by_league(df, referee_stats)
save("referee.json", referee)
pd.DataFrame(referee["All"])

saved referee.json


,group,matches,fouls,yellows,reds_per_100
0,With fans,13941,0.32,0.27,1.2
1,Empty stadiums,2122,-0.23,-0.01,1.3


## Section 6: the players

Extra shots and shots on target for the home team (home − away), with fans vs without.

In [6]:
def player_stats(data):
    rows = []
    for label, crowd in [("With fans", 1), ("Empty stadiums", 0)]:
        part = data[data["crowd"] == crowd]
        rows.append({
            "group": label,
            "shots": round(part["shot_diff"].mean(), 2),
            "shots_on_target": round(part["sot_diff"].mean(), 2),
        })
    return rows

players = by_league(df, player_stats)
save("players.json", players)
pd.DataFrame(players["All"])

saved players.json


,group,shots,shots_on_target
0,With fans,2.56,0.89
1,Empty stadiums,1.35,0.52


## Section 7: crowd size

The EDA used 5 groups, but the small ones (like 1–25% full, only 445 matches) jump around by chance. The website uses 3 groups with enough matches each.

In [7]:
# 3 groups for the website: the 5 groups in 03_eda had small, noisy groups (445 matches at 1–25%)
bins = [-0.01, 0, 0.5, 1]
labels = ["Empty", "Under half full", "Half full or more"]
sized = df.dropna(subset=["attendance_pct"]).copy()
sized["fill"] = pd.cut(sized["attendance_pct"], bins=bins, labels=labels)

crowd_size = []
for fill, part in sized.groupby("fill", observed=True):
    crowd_size.append({
        "fill": fill,
        "matches": int(len(part)),
        "home_win": round((part["result"] == "H").mean() * 100, 1),
        "home_points": round(part["home_points"].mean(), 2),
    })

save("crowd_size.json", crowd_size)
pd.DataFrame(crowd_size)

saved crowd_size.json


,fill,matches,home_win,home_points
0,Empty,2122,39.9,1.45
1,Under half full,1558,42.9,1.54
2,Half full or more,12381,44.3,1.58


## Section 8: your team

Each team's home record with fans and in empty stadiums. Only teams with at least 5 home ghost games are included, so the numbers aren't based on one or two matches.

In [8]:
teams = {}
for (league, team), home in df.groupby(["league", "home_team"]):
    fans = home[home["crowd"] == 1]
    ghost = home[home["crowd"] == 0]
    if len(ghost) < 5:
        continue
    teams.setdefault(league, []).append({
        "team": team,
        "fans_matches": int(len(fans)),
        "fans_win": round((fans["result"] == "H").mean() * 100, 1),
        "fans_points": round(fans["home_points"].mean(), 2),
        "ghost_matches": int(len(ghost)),
        "ghost_win": round((ghost["result"] == "H").mean() * 100, 1),
        "ghost_points": round(ghost["home_points"].mean(), 2),
    })

for league in teams:
    teams[league] = sorted(teams[league], key=lambda t: t["team"])

save("teams.json", teams)
print("teams:", sum(len(t) for t in teams.values()))

saved teams.json
teams: 108


## Section 9: the match simulator

Copy the model export from `04_model`.

In [9]:
shutil.copy("../data/clean/model_export.json", OUT / "model_export.json")
print("copied model_export.json")
sorted(p.name for p in OUT.iterdir())

copied model_export.json


['attendance.json',
 'crowd_size.json',
 'model_export.json',
 'players.json',
 'referee.json',
 'results.json',
 'summary.json',
 'teams.json']